# KAT-VAD v2 — **Final: the sealed sets, opened once** (Amendment 10, §19 P1–P8)

Pre-registration: `core/docs/v2/PREREG_ADDENDUM.md` **§19** (signed 2026-10-06) and **§20** (5 featureless DoTA-eval
clips), on top of §18 (E3) and §17 (O1′).
E3 adopted **A3** (`F` = A0); this notebook **reports**, it never re-decides (P1). **No training.** It scores the 20
E3 checkpoints (step 1740) on the sets E3 never read:

| step | what | sealed? | GPU? |
|---|---|---|---|
| 0 | setup; §19 must be signed; E3 decision must be A3 / A0 | no | — |
| 1 | **P7 gate**: `final_readout --gate-only` reproduces `RESULTS_E3.md` §5 on DoTA-CAP-dev. **FAIL → stop** | no | no |
| 2 | preflight (features exist; §20's 5 featureless DoTA-eval clips, exactly) + stage; **`OPENED.json`** marker | — | no |
| 3 | bake the eval side with the T2-train stats (`build_v2_inputs apply`, never `fit`) | features only | no |
| 4 | protocol B `--final`: A0–A3 on `dota_cap_eval` (560), A0/A1 on `dota_eval` (695 / 700, §20) | **yes** | yes |
| 5 | `p_T2` `--final` on both eval splits | **yes** | no |
| 6 | T2-test (1,106 windows): `v2_guard_test` per seed, A0–A3 | **yes** | yes |
| 7 | `final_readout` → `final_readout.md` → **stop** | — | no |

**Once means once.** Every step skips an output that already exists on Drive, so a disconnect resumes; nothing is
ever re-scored with other settings. Do not delete `v2_final/` to "try again". All outputs go to
`Thesis-V2/outputs/v2_final/`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read: data/, cache/ (CLIP, VideoMAE)
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read: kat-vad/, cache/v2/, outputs/v2_e3; write: outputs/v2_final, cache/v2/
os.environ['PROJECT_ROOT']       = DRIVE_V2
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_V2}/outputs'

os.environ['REPO'] = f'{DRIVE_V2}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v2 there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert (REPO / 'core/tools/final_readout.py').is_file(), 'this upload predates the Final harness -- re-upload branch v2'

DATASET    = constants.DADA_ORIGIN_DATASET               # 'DADA2000_orig'
SEEDS      = list(constants.V2_E3_SEEDS)                 # M1: 2024-2028
ARMS       = list(constants.V2_E3_ARMS)
CRN        = 'R2'                                        # D11
ENC        = constants.VIDEOMAE_ENCODER_S                # E2(d) pick
STEP       = 1740                                        # J10 step of every E3 run
T2_TEST_WINDOWS = 1106

T2         = constants.DATA_ROOT / DATASET               # PARENT T2 dir: its evaluation file is T2-test
CLIP_DIR   = constants.CLIP_CACHE_DIR / DATASET
DOTA       = constants.DATA_ROOT / 'DoTA'
LABELS     = DOTA / 'labels_s8'                          # defs.json (class names)
DOTA_S1    = constants.CLIP_CACHE_DIR / 'DoTA_s1_ncc'
S_CAP      = constants.VIDEO_CACHE_DIR / ENC / f'{constants.DOTA_CAP_DATASET}_s1_squash'
V2_DATA    = Path(DRIVE_V2) / 'data' / f'{DATASET}_v2'
V2_CACHE   = Path(DRIVE_V2) / 'cache' / 'v2'
TAG        = {'A1': f'A1_{CRN}', 'A2': 'A2_S', 'A3': f'A3_{CRN}_S'}
BAKED_T2   = {arm: V2_CACHE / TAG[arm] / DATASET for arm in TAG}               # fit dirs = the T2 stats (E3's)
A1_EVAL    = V2_CACHE / TAG['A1'] / 'DoTA_eval_s3_from_s1'                     # dota_eval (covers dota_cap_eval)
CAP_EVAL_BAKED = {arm: V2_CACHE / TAG[arm] / 'DoTA_CAP_eval_s3_from_s1' for arm in ('A2', 'A3')}
E3         = constants.OUTPUT_ROOT / 'v2_e3'             # E3's runs + REPORTS (read only here)
FINAL      = constants.OUTPUT_ROOT / 'v2_final'          # everything this notebook writes
OPENED     = FINAL / 'OPENED.json'
LOCAL = Path('/content/final')
LOCAL.mkdir(parents=True, exist_ok=True)
for name in ('E3', 'FINAL', 'V2_DATA', 'S_CAP'):
    print(f'{name:8s}', globals()[name])

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu
python -c "import transformers, av, einops, faiss; print('transformers', transformers.__version__)"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- training will be unusably slow'
df -h /content | tail -1

In [ ]:
import json
import shutil
import subprocess
import time

import numpy as np
import torch
import yaml

ENV = dict(os.environ)


def run(cmd, log=None, quiet_steps=False):
    """Stream a child's output here (and to ``log``); raise with its tail on failure."""
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                if not (quiet_steps and 'global_step' in line):
                    print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    """Copy a Drive dir (or only ``ids`` + its manifests) / file to VM disk once (bit-identical; C2)."""
    if dst.exists() and ids is None:
        return dst
    t0 = time.time()
    if src.is_dir() and ids is None:
        shutil.copytree(src, dst)
    elif src.is_dir():
        dst.mkdir(parents=True, exist_ok=True)
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for m in src.glob('*.json'):
            shutil.copy2(m, dst / m.name)
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    """Copy a VM-local result to Drive and fsync every file (C10)."""
    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True)
        files = [p for p in dst.rglob('*') if p.is_file()]
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        files = [dst]
    for p in files:
        with open(p, 'rb+') as fh:
            os.fsync(fh.fileno())


def ckpt_step(path):
    if not path.exists():
        return 0
    return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))


def metrics_last(path):
    lines = [l for l in path.read_text().splitlines() if l.strip()] if path.exists() else []
    return json.loads(lines[-1]) if lines else None

## 0. Signed rules, E3's decision, E3's checkpoints (nothing sealed is read)

In [ ]:
from core.tools import e3_readout, final_readout, position_prior, protocol_b_eval, v2_dataset, v2_guard_test
from core.tools.v2_diagnostics import DIAG_JSON

addendum = (REPO / 'core/docs/v2/PREREG_ADDENDUM.md').read_text(encoding='utf-8')
assert '## 19. Amendment 10' in addendum, 'upload predates §19'
s19 = addendum.split('## 19. Amendment 10', 1)[1]
assert 'Authorized by the user on' in s19 and 'Not in force until' not in s19, \
    '§19 is not signed in this upload -- sign + commit + re-upload before opening anything'
decision = json.loads((E3 / e3_readout.REPORTS / e3_readout.E3_READOUT_JSON).read_text())['decision']
assert (decision['adopt'], decision['F']) == (constants.V2_FINAL_ADOPTED, constants.V2_FINAL_F_ARM), decision


def run_dir(arm, seed):
    return E3 / arm / f's{seed}'


def ckpt(arm, seed):
    return run_dir(arm, seed) / 'checkpoint_last.pt'


for seed in SEEDS:                       # J10 on every E3 checkpoint, and the diag O1' reads beside it
    for arm in ARMS:
        step, last = ckpt_step(ckpt(arm, seed)), metrics_last(run_dir(arm, seed) / 'metrics.jsonl')
        assert last is not None and step == last['global_step'] == STEP, f'{arm} s{seed}: J10 (ckpt {step})'
        assert (run_dir(arm, seed) / e3_readout.DIAG_DIR / DIAG_JSON).exists(), f'{arm} s{seed}: no E3 diag'
print('20 E3 checkpoints at step', STEP, '(J10 ok); E3 decision', decision)

## 1. P7 gate — dev only. FAIL → stop, open nothing

In [ ]:
out = E3 / e3_readout.REPORTS
run([sys.executable, '-m', 'core.tools.final_readout', '--e3-dir', E3, '--gate-only'])
gate = json.loads((out / final_readout.GATE_JSON).read_text())
assert gate['pass'], f'P7 gate FAILS -- STOP, paste this to Claude, open nothing: {gate["failed"]}'
print(f'P7 gate PASS: {len(gate["checks"])} dev means == RESULTS_E3.md §5')

## 2. Preflight + stage; the sets are opened here (`OPENED.json`)

In [ ]:
from core.data.v2_splits import load_split

dev = load_split(constants.V2_SPLIT_DOTA_DEV)
ev = load_split(constants.V2_SPLIT_DOTA_EVAL, final=True)          # §19 P2: the only place final=True is used
cap_ev = load_split(constants.V2_SPLIT_DOTA_CAP_EVAL, final=True)
assert (len(ev), len(cap_ev)) == (700, 560), (len(ev), len(cap_ev))
assert set(cap_ev) <= set(ev) and not set(ev) & set(dev), 'split algebra broken'
missing = [v for v in cap_ev if not (S_CAP / f'{v}.npy').exists()]
assert not missing, f'{len(missing)} dota_cap_eval clips have no V2-S features in {S_CAP}: {missing[:5]}'
# §20 R1: exactly the 5 known clips lack CLIP features (v1 FUSE unzip); anything else -> STOP
no_clip = sorted(v for v in ev if not (DOTA_S1 / f'{v}.npy').exists())
assert no_clip == sorted(constants.V2_FINAL_DOTA_EVAL_NO_FEATURES), \
    f'clips without CLIP features {no_clip} != the §20 list -- STOP, paste to Claude'
assert not set(no_clip) & set(cap_ev), 'a featureless clip is in dota_cap_eval'
s20 = addendum.split('## 20. Amendment 11', 1)[1] if '## 20. Amendment 11' in addendum else ''
assert 'Authorized by the user on' in s20 and 'Not in force until' not in s20, \
    '§20 (Amendment 11) is not signed in this upload -- sign + re-upload'
ev_scored = [v for v in ev if v not in no_clip]          # 695: the DoTA-eval clips that can be scored
for arm, d in BAKED_T2.items():
    m = json.loads((d / constants.V2_INPUT_MANIFEST_FILENAME).read_text())
    assert m['arm'] == arm and m['crn'] == ('none' if arm == 'A2' else CRN), m

if not OPENED.exists():
    FINAL.mkdir(parents=True, exist_ok=True)
    OPENED.write_text(json.dumps({
        'opened_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
        'addendum': '§19 Amendment 10', 'adopted': decision,
        'sets': ['dota_eval', 'dota_cap_eval', 't2_test'],
    }, indent=2))
    print('sealed sets OPENED now:', OPENED)
else:
    print('resuming -- sets were opened on', json.loads(OPENED.read_text())['opened_utc'])

L_V2 = stage(V2_DATA, LOCAL / 'stage' / 'corpus_v2')
L_DOTA_S1 = stage(DOTA_S1, LOCAL / 'stage' / 'DoTA_s1_ncc_eval', ev_scored)
L_S_CAP = stage(S_CAP, LOCAL / 'stage' / 's_cap_eval', cap_ev)
L_DOTA_LABELS = stage(LABELS / constants.DEFS_FILENAME,
                      LOCAL / 'stage' / 'dota_labels' / constants.DEFS_FILENAME).parent
print(f'DoTA-eval {len(ev_scored)} / {len(ev)} clips scorable (§20), DoTA-CAP-eval {len(cap_ev)}')

## 3. Bake the eval side (`apply` with E3's T2-train stats; never `fit`)

In [ ]:
def bake(stats_dir, ids, out_drive, video_dir=None):
    if (out_drive / constants.V2_INPUT_MANIFEST_FILENAME).exists():
        print(f'reusing {out_drive}')
        return stage(out_drive, LOCAL / 'stage' / out_drive.parent.name / out_drive.name)
    ids_file = LOCAL / f'{out_drive.parent.name}_{out_drive.name}.txt'
    ids_file.write_text(''.join(f'{v}\n' for v in ids))
    local = LOCAL / 'bake' / out_drive.parent.name / out_drive.name
    run([sys.executable, '-m', 'core.tools.build_v2_inputs', 'apply', '--stats-dir', stats_dir,
         '--clip-dir', L_DOTA_S1, *(['--video-dir', video_dir] if video_dir else []),
         '--stride', constants.V2_E1_STRIDE_BC, '--ids-file', ids_file, '--out-dir', local])
    publish(local, out_drive)
    return local


L_A1_EVAL = bake(BAKED_T2['A1'], ev_scored, A1_EVAL)
L_CAP_EVAL = {arm: bake(BAKED_T2[arm], cap_ev, CAP_EVAL_BAKED[arm], L_S_CAP) for arm in CAP_EVAL_BAKED}
for arm, d in {'A1': L_A1_EVAL, **L_CAP_EVAL}.items():   # same T2 stats as E3's dev bakes (M1)
    m = json.loads((d / constants.V2_INPUT_MANIFEST_FILENAME).read_text())
    t2 = json.loads((BAKED_T2[arm] / constants.V2_INPUT_MANIFEST_FILENAME).read_text())
    assert m['train_ids_sha1'] == t2['train_ids_sha1'], f'{arm}: eval bake on another train split'
    print(f'{arm}: baked {m["baked_ids"]} clips from the T2-train stats ({t2["train_ids_sha1"][:12]})')

## 4. Protocol B `--final` (P2)

In [ ]:
SPLIT_INPUT = {
    ('A0', constants.V2_SPLIT_DOTA_EVAL): L_DOTA_S1, ('A0', constants.V2_SPLIT_DOTA_CAP_EVAL): L_DOTA_S1,
    ('A1', constants.V2_SPLIT_DOTA_EVAL): L_A1_EVAL, ('A1', constants.V2_SPLIT_DOTA_CAP_EVAL): L_A1_EVAL,
    ('A2', constants.V2_SPLIT_DOTA_CAP_EVAL): L_CAP_EVAL['A2'],
    ('A3', constants.V2_SPLIT_DOTA_CAP_EVAL): L_CAP_EVAL['A3'],
}
for (arm, split), input_dir in SPLIT_INPUT.items():
    out = e3_readout.pb_dir(FINAL, arm, split)
    if (out / protocol_b_eval.CLIP_SCORES_NPZ).exists():
        print(f'{arm}/{split}: already scored -- not re-scored')
        continue
    runs = [x for s in SEEDS for x in ('--run', f's{s}', ckpt(arm, s))]
    run([sys.executable, '-m', 'core.tools.protocol_b_eval', *runs, '--final', '--exclude-featureless',
         '--input-dir', input_dir, '--s1-dir', L_DOTA_S1,
         '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt',
         '--data-dir', L_DOTA_LABELS, '--split', split, '--out-dir', out, '--device', 'auto'],
        log=LOCAL / f'{arm}_{split}_pb.log')
print('protocol B done for', len(SPLIT_INPUT), 'arm x split pairs')

## 5. `p_T2` on the eval splits (`--final`; the fit sees no DoTA label)

In [ ]:
for split in (constants.V2_SPLIT_DOTA_EVAL, constants.V2_SPLIT_DOTA_CAP_EVAL):
    out = FINAL / e3_readout.REPORTS / e3_readout.PRIOR_DIR / split
    if not (out / position_prior.PRIOR_NPZ).exists():
        run([sys.executable, '-m', 'core.tools.position_prior', '--final', '--exclude-featureless',
             '--data-dir', L_V2,
             '--s1-dir', L_DOTA_S1, '--metadata', DOTA / 'metadata_val.json',
             '--split-file', DOTA / 'val_split.txt', '--split', split, '--out-dir', out])
    print((out / position_prior.READOUT_MD).read_text())

## 6. T2-test, per seed — `v2_guard_test` (A0–A3, O1′ vs A0 of the same seed)

In [ ]:
L_T2 = stage(T2, LOCAL / 'stage' / 'corpus')
test = json.loads((L_T2 / constants.FRAME_LABELS_TEST_FILENAME).read_text())
assert len(test) == T2_TEST_WINDOWS, f'{len(test)} T2-test windows, expected {T2_TEST_WINDOWS}'
INPUT = {'A0': CLIP_DIR, **BAKED_T2}           # v2_guard_test's hard gate checks every T2-test source is there
L_IN = {arm: stage(d, LOCAL / 'stage' / f'{arm}_T2') for arm, d in INPUT.items()}
for seed in SEEDS:
    out = FINAL / final_readout.T2_TEST_DIR / f's{seed}'
    if (out / v2_guard_test.GUARD_JSON).exists():
        print(f's{seed}: already scored -- not re-scored')
        continue
    cmd = [sys.executable, '-m', 'core.tools.v2_guard_test', '--data-dir', L_T2, '--a0', 'A0',
           '--out-dir', LOCAL / 't2_test' / f's{seed}', '--device', 'auto']
    for arm in ARMS:
        cmd += ['--arm', arm, ckpt(arm, seed), L_IN[arm],
                '--val-diag', arm, run_dir(arm, seed) / e3_readout.DIAG_DIR / DIAG_JSON]
    run(cmd, log=LOCAL / f't2_test_s{seed}.log')
    publish(LOCAL / 't2_test' / f's{seed}', out)
    r = json.loads((out / v2_guard_test.GUARD_JSON).read_text())
    assert all(r['runs'][a]['windows'] == T2_TEST_WINDOWS for a in ARMS), f's{seed}: window count'
print('T2-test scored for', len(SEEDS), 'seeds x 4 arms')

## 7. Final read-out (P3–P5, P8)

In [ ]:
run([sys.executable, '-m', 'core.tools.final_readout', '--e3-dir', E3, '--final-dir', FINAL])
print((FINAL / e3_readout.REPORTS / final_readout.FINAL_READOUT_MD).read_text())

## 8. Record — then **stop**

Paste `v2_final/REPORTS/final_readout.md` back to Claude (and `final_readout.json` if asked). The adoption is E3's
(P1); the sentences under "Position control" are fixed by §19 P5. Nothing is re-run, re-scored or re-tuned after
reading it. `T2_test/s*/o1prime_t2test.md` also prints the pilot's Q5 wording — ignore it here; the Final's T2-test
table is the one in `final_readout.md`.